# LifeMate – predict notebook (timing)
Run All. Writes `/result/jupyter_submission.jsonl` (same format as `submission.jsonl`, with `time` in ms). Fully offline.

In [ ]:
# Step 1 - fixed seed
import os, sys, random
import numpy as np
import torch
sys.path.insert(0, "/code")

def seed_everything(seed=2026):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(2026)

In [ ]:
# Step 2 - load ASR + LLM from local disk (no external API, no internet)
import time
import zac_runtime as rt

t0 = time.perf_counter()
runtime = rt.load_models(rt.DATA_ROOT, rt.MODELS_DIR)   # PhoWhisper-large (fine-tuned) + Qwen3-4B (LoRA merged)
print(f"models loaded in {time.perf_counter() - t0:.1f}s; split={runtime.split}")

In [ ]:
# Step 3 - read the test cases
import json
from pathlib import Path

ROOT = rt.DATA_ROOT
SPLIT = runtime.split
sys.path.insert(0, str(ROOT / SPLIT))        # zaic_tools of the split, from /data (not from the image)
from zaic_tools import ToolEnv                # noqa: F401  (each sample gets exactly one ToolEnv, created in run_sample)

test_cases = rt.read_items(ROOT, SPLIT)
print(len(test_cases), "test cases")

In [ ]:
# Step 4 - predict one test case at a time, record time (ms) per case
all_result = []
for item in test_cases:
    record = rt.run_sample(item, runtime)      # ASR -> LLM -> env.call(...) -> ... ; time measured inside
    all_result.append(record)

Path("/result").mkdir(parents=True, exist_ok=True)
with open("/result/jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("done:", len(all_result), "rows; max ms =", max((r["time"] for r in all_result), default=0))